In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# --- Same split as the simple run above (train on last 4 seasons, test on 25-26),
#     but using predict_proba() to get real probabilities instead of a hard label,
#     converted to odds (1/p) and compared against the bookmakers' average odds. ---
all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons_with_bookies.csv")

train_seasons = ["21-22", "22-23", "23-24", "24-25"]
test_season = "25-26"

train_df = all_df[all_df["Season"].isin(train_seasons)]
test_df = all_df[all_df["Season"] == test_season].copy()

# Exclude the usual identifiers/leakage columns AND the bookmaker odds columns
# themselves -- those are what we're comparing against, not features to train on.
drop_cols = [
    "Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season",
    "AvgHomeOdds", "AvgDrawOdds", "AvgAwayOdds", "NumBookies"
]

X_train = train_df.drop(columns=drop_cols).fillna(0)
y_train = train_df["FTR"]

X_test = test_df.drop(columns=drop_cols).fillna(0)

print(f"Train: {train_seasons} -> {X_train.shape[0]} matches, {X_train.shape[1]} features")
print(f"Test: {test_season} -> {X_test.shape[0]} matches")

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)
rf.fit(X_train, y_train)

# --- Convert predicted probabilities into "model odds" ---
proba = rf.predict_proba(X_test)
model_odds = 1 / np.clip(proba, 1e-6, None)  # clip guards against a divide-by-zero on a 0% class

class_to_odds_col = {"H": "ModelHomeOdds", "D": "ModelDrawOdds", "A": "ModelAwayOdds"}
odds_cols_ordered = [class_to_odds_col[c] for c in rf.classes_]
model_odds_df = pd.DataFrame(model_odds, columns=odds_cols_ordered, index=test_df.index)

comparison = pd.concat([
    test_df[["Date", "HomeTeam", "AwayTeam", "FTR", "AvgHomeOdds", "AvgDrawOdds", "AvgAwayOdds"]],
    model_odds_df[["ModelHomeOdds", "ModelDrawOdds", "ModelAwayOdds"]]
], axis=1)

# The market's odds carry a built-in margin (overround), but the model's
# probabilities sum to exactly 1 -- de-vig the market odds too so the
# comparison isn't just "model odds are bigger because they have no margin".
overround = 1 / comparison["AvgHomeOdds"] + 1 / comparison["AvgDrawOdds"] + 1 / comparison["AvgAwayOdds"]
comparison["FairMarketHomeOdds"] = 1 / (1 / comparison["AvgHomeOdds"] / overround)
comparison["FairMarketDrawOdds"] = 1 / (1 / comparison["AvgDrawOdds"] / overround)
comparison["FairMarketAwayOdds"] = 1 / (1 / comparison["AvgAwayOdds"] / overround)

pd.set_option("display.width", 160)
print("\nSample comparison (first 10 matches):")
print(comparison.head(10).round(2))

print("\n=== Model odds vs. market odds ===")
for outcome, model_col, market_col, fair_col in [
    ("Home", "ModelHomeOdds", "AvgHomeOdds", "FairMarketHomeOdds"),
    ("Draw", "ModelDrawOdds", "AvgDrawOdds", "FairMarketDrawOdds"),
    ("Away", "ModelAwayOdds", "AvgAwayOdds", "FairMarketAwayOdds"),
]:
    mean_model = comparison[model_col].mean()
    mean_market = comparison[market_col].mean()
    mean_fair = comparison[fair_col].mean()
    mae_vs_market = (comparison[model_col] - comparison[market_col]).abs().mean()
    mae_vs_fair = (comparison[model_col] - comparison[fair_col]).abs().mean()
    corr = comparison[model_col].corr(comparison[market_col])
    print(
        f"{outcome:5s}: mean model={mean_model:5.2f}  mean market={mean_market:5.2f}  "
        f"mean fair-market={mean_fair:5.2f}  MAE(vs market)={mae_vs_market:.2f}  "
        f"MAE(vs fair)={mae_vs_fair:.2f}  corr={corr:.3f}"
    )



Train: ['21-22', '22-23', '23-24', '24-25'] -> 1439 matches, 41 features
Test: 25-26 -> 360 matches

Sample comparison (first 10 matches):
            Date       HomeTeam        AwayTeam FTR  AvgHomeOdds  AvgDrawOdds  AvgAwayOdds  ModelHomeOdds  ModelDrawOdds  ModelAwayOdds  FairMarketHomeOdds  \
5033  2025-08-30        Chelsea          Fulham   H         1.55         4.39         5.60           1.90           3.88           4.65                1.63   
5034  2025-08-30     Man United         Burnley   H         1.35         5.26         8.34           2.15           3.68           3.81                1.42   
5035  2025-08-30     Sunderland       Brentford   H         2.94         3.29         2.47           4.38           3.46           2.07                3.08   
5036  2025-08-30      Tottenham     Bournemouth   A         1.74         4.04         4.35           2.46           3.99           2.92                1.83   
5037  2025-08-30         Wolves         Everton   A         2.67  

In [2]:
import numpy as np
from sklearn.metrics import log_loss
from sklearn.preprocessing import label_binarize

# --- Score the model's probabilities and the market's (de-vigged) probabilities
#     against what actually happened -- this is the real "who's the better
#     forecaster" test, not just an odds-to-odds comparison. ---

classes_order = rf.classes_  # ['A', 'D', 'H'], same order `proba`'s columns are in
y_true = test_df["FTR"].values
y_true_onehot = label_binarize(y_true, classes=classes_order)

# --- Model ---
model_log_loss = log_loss(y_true, proba, labels=classes_order)
model_brier = ((proba - y_true_onehot) ** 2).sum(axis=1).mean()

# --- Market, de-vigged so it's a fair fight (probabilities sum to 1, like the model's) ---
class_to_fair_col = {"H": "FairMarketHomeOdds", "D": "FairMarketDrawOdds", "A": "FairMarketAwayOdds"}
fair_market_proba = np.column_stack([1 / comparison[class_to_fair_col[c]] for c in classes_order])

market_log_loss = log_loss(y_true, fair_market_proba, labels=classes_order)
market_brier = ((fair_market_proba - y_true_onehot) ** 2).sum(axis=1).mean()

print("=== Log loss (lower is better) ===")
print(f"Model:  {model_log_loss:.4f}")
print(f"Market: {market_log_loss:.4f}")

print("\n=== Brier score (lower is better) ===")
print(f"Model:  {model_brier:.4f}")
print(f"Market: {market_brier:.4f}")

winner = "Model" if model_log_loss < market_log_loss else "Market"
print(f"\n{winner} is the better-calibrated forecaster on {test_season} by log loss.")

=== Log loss (lower is better) ===
Model:  1.0337
Market: 1.0195

=== Brier score (lower is better) ===
Model:  0.6230
Market: 0.6132

Market is the better-calibrated forecaster on 25-26 by log loss.


In [12]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import label_binarize
from scipy import stats

# --- Walk-forward across every available season, with a validation season
#     carved out of each training window so overfitting is visible fold-by-fold
#     (train log loss vs. validation log loss), not just inferred after the fact.
#
#     WINDOW and VAL_SEASONS are the ONLY place to change these -- they're passed
#     through as keyword args below, not re-hardcoded at the call site, to avoid
#     the earlier bug where the default said one thing and the call site silently
#     used another. ---

WINDOW = 4       # total seasons of history pulled into each fold
VAL_SEASONS = 1   # how many of the most-recent seasons in that window are held out for validation

all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons_with_bookies.csv")
all_df = all_df.reset_index(drop=True)

drop_cols = [
    "Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season",
    "AvgHomeOdds", "AvgDrawOdds", "AvgAwayOdds", "NumBookies"
]


class WalkForwardValidator:

    def __init__(self, seasons):
        self.seasons = seasons

    def split(self, meta, window, val_seasons=1):
        unique_seasons = sorted(meta["Season"].unique())
        for i in range(window, len(unique_seasons)):
            window_seasons = unique_seasons[i - window:i]
            train_seasons = window_seasons[:-val_seasons]
            val_season_list = window_seasons[-val_seasons:]
            test_season = unique_seasons[i]

            train_idx = meta[meta["Season"].isin(train_seasons)].index
            val_idx = meta[meta["Season"].isin(val_season_list)].index
            test_idx = meta[meta["Season"] == test_season].index

            yield (train_idx, val_idx, test_idx, train_seasons, val_season_list, test_season)


def eval_probs(rf, X, y, classes_order):
    """Log loss + Brier score for a fitted model against a labeled set."""
    proba = rf.predict_proba(X)
    y_onehot = label_binarize(y, classes=classes_order)
    eps = 1e-15
    ll = -np.log(np.clip((proba * y_onehot).sum(axis=1), eps, 1)).mean()
    brier = ((proba - y_onehot) ** 2).sum(axis=1).mean()
    return ll, brier


validator = WalkForwardValidator(seasons=sorted(all_df["Season"].unique()))

fold_rows = []
all_model_ll, all_market_ll = [], []
all_model_brier, all_market_brier = [], []

for train_idx, val_idx, test_idx, train_seasons, val_season_list, test_season in validator.split(
    all_df, window=WINDOW, val_seasons=VAL_SEASONS
):

    train_df = all_df.loc[train_idx]
    val_df = all_df.loc[val_idx]
    test_df = all_df.loc[test_idx]

    X_train = train_df.drop(columns=drop_cols).fillna(0)
    y_train = train_df["FTR"]
    X_val = val_df.drop(columns=drop_cols).fillna(0)
    y_val = val_df["FTR"]
    X_test = test_df.drop(columns=drop_cols).fillna(0)

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )
    rf.fit(X_train, y_train)
    classes_order = rf.classes_

    # --- Train / validation log loss & Brier -- the overfitting check ---
    train_ll, train_brier = eval_probs(rf, X_train, y_train, classes_order)
    val_ll, val_brier = eval_probs(rf, X_val, y_val, classes_order)

    # --- Test set: model vs. de-vigged market, same as before ---
    proba = rf.predict_proba(X_test)
    overround = 1 / test_df["AvgHomeOdds"] + 1 / test_df["AvgDrawOdds"] + 1 / test_df["AvgAwayOdds"]
    fair = {
        "H": (1 / test_df["AvgHomeOdds"]) / overround,
        "D": (1 / test_df["AvgDrawOdds"]) / overround,
        "A": (1 / test_df["AvgAwayOdds"]) / overround,
    }
    fair_market_proba = np.column_stack([fair[c].values for c in classes_order])

    y_true = test_df["FTR"].values
    y_onehot = label_binarize(y_true, classes=classes_order)

    eps = 1e-15
    model_ll = -np.log(np.clip((proba * y_onehot).sum(axis=1), eps, 1))
    market_ll = -np.log(np.clip((fair_market_proba * y_onehot).sum(axis=1), eps, 1))
    model_brier = ((proba - y_onehot) ** 2).sum(axis=1)
    market_brier = ((fair_market_proba - y_onehot) ** 2).sum(axis=1)

    all_model_ll.append(model_ll)
    all_market_ll.append(market_ll)
    all_model_brier.append(model_brier)
    all_market_brier.append(market_brier)

    fold_rows.append({
        "test_season": test_season,
        "train_seasons": train_seasons,
        "val_season": val_season_list,
        "n_test": len(test_df),
        "train_log_loss": train_ll,
        "val_log_loss": val_ll,
        "test_log_loss": model_ll.mean(),
        "market_log_loss": market_ll.mean(),
        "train_brier": train_brier,
        "val_brier": val_brier,
        "test_brier": model_brier.mean(),
        "market_brier": market_brier.mean(),
    })

    print(
        f"{test_season} (train {train_seasons[0]}..{train_seasons[-1]}, val {val_season_list[0]}): "
        f"log loss train={train_ll:.4f} val={val_ll:.4f} test={model_ll.mean():.4f} market={market_ll.mean():.4f}"
    )

fold_df = pd.DataFrame(fold_rows)
print("\n=== Per-fold summary (overfitting check: train vs. val vs. test) ===")
print(fold_df[["test_season", "train_log_loss", "val_log_loss", "test_log_loss", "market_log_loss"]].round(4))

# --- Pool every match across every fold's TEST set for the model-vs-market test ---
model_ll_all = np.concatenate(all_model_ll)
market_ll_all = np.concatenate(all_market_ll)
model_brier_all = np.concatenate(all_model_brier)
market_brier_all = np.concatenate(all_market_brier)

print(f"\n=== Pooled across all {len(fold_df)} folds ({len(model_ll_all)} test matches) ===")
print(f"Model log loss:  {model_ll_all.mean():.4f}")
print(f"Market log loss: {market_ll_all.mean():.4f}")
print(f"Model Brier:     {model_brier_all.mean():.4f}")
print(f"Market Brier:    {market_brier_all.mean():.4f}")

print(f"\nMean train log loss:      {fold_df['train_log_loss'].mean():.4f}")
print(f"Mean validation log loss: {fold_df['val_log_loss'].mean():.4f}")
print(
    f"Gap (val - train): {fold_df['val_log_loss'].mean() - fold_df['train_log_loss'].mean():.4f}  "
    "(large gap = overfitting -- the model fits the training seasons far better than unseen ones)"
)

t_stat, p_value = stats.ttest_rel(model_ll_all, market_ll_all)
print(f"\nPaired t-test (log loss, model vs market): t={t_stat:.3f}, p={p_value:.6f}")

diff = model_ll_all - market_ll_all
rng = np.random.default_rng(42)
n = len(diff)
boot_means = np.array([diff[rng.integers(0, n, n)].mean() for _ in range(10000)])
ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
print(f"Bootstrap 95% CI for mean log-loss diff (model - market): [{ci_low:.4f}, {ci_high:.4f}]")

model_wins = (model_ll_all < market_ll_all).sum()
market_wins = (market_ll_all < model_ll_all).sum()
print(f"\nModel better on {model_wins}/{n} matches, market better on {market_wins})/{n} matches")
print(f"Model better on {model_wins/n:.2%} of matches, market better on {market_wins/n:.2%} of matches")

15-16 (train 11-12..13-14, val 14-15): log loss train=0.7728 val=1.0033 test=1.0433 market=1.0301
16-17 (train 12-13..14-15, val 15-16): log loss train=0.7694 val=1.0415 test=0.9761 market=0.9088
17-18 (train 13-14..15-16, val 16-17): log loss train=0.7888 val=0.9736 test=0.9890 market=0.9398
18-19 (train 14-15..16-17, val 17-18): log loss train=0.7740 val=0.9927 test=0.9512 market=0.8983
19-20 (train 15-16..17-18, val 18-19): log loss train=0.7809 val=0.9421 test=1.0128 market=0.9759
20-21 (train 16-17..18-19, val 19-20): log loss train=0.7550 val=1.0160 test=1.0418 market=1.0096
21-22 (train 17-18..19-20, val 20-21): log loss train=0.7769 val=1.0300 test=1.0028 market=0.9425
22-23 (train 18-19..20-21, val 21-22): log loss train=0.7831 val=1.0108 test=1.0135 market=0.9617
23-24 (train 19-20..21-22, val 22-23): log loss train=0.7985 val=1.0075 test=0.9788 market=0.9168
24-25 (train 20-21..22-23, val 23-24): log loss train=0.7946 val=0.9822 test=1.0060 market=0.9820
25-26 (train 21-22..

In [11]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import label_binarize
from scipy import stats

# --- Walk-forward across every available season, with a validation season
#     carved out of each training window so overfitting is visible fold-by-fold
#     (train log loss vs. validation log loss), not just inferred after the fact.
#
#     WINDOW and VAL_SEASONS are the ONLY place to change these -- they're passed
#     through as keyword args below, not re-hardcoded at the call site, to avoid
#     the earlier bug where the default said one thing and the call site silently
#     used another. ---

WINDOW = 3        # total seasons of history pulled into each fold
VAL_SEASONS = 1   # how many of the most-recent seasons in that window are held out for validation

all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons_with_bookies.csv")
all_df = all_df.reset_index(drop=True)

drop_cols = [
    "Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season",
    "AvgHomeOdds", "AvgDrawOdds", "AvgAwayOdds", "NumBookies"
]


class WalkForwardValidator:

    def __init__(self, seasons):
        self.seasons = seasons

    def split(self, meta, window, val_seasons=1):
        unique_seasons = sorted(meta["Season"].unique())
        for i in range(window, len(unique_seasons)):
            window_seasons = unique_seasons[i - window:i]
            train_seasons = window_seasons[:-val_seasons]
            val_season_list = window_seasons[-val_seasons:]
            test_season = unique_seasons[i]

            train_idx = meta[meta["Season"].isin(train_seasons)].index
            val_idx = meta[meta["Season"].isin(val_season_list)].index
            test_idx = meta[meta["Season"] == test_season].index

            yield (train_idx, val_idx, test_idx, train_seasons, val_season_list, test_season)


def eval_probs(rf, X, y, classes_order):
    """Log loss + Brier score for a fitted model against a labeled set."""
    proba = rf.predict_proba(X)
    y_onehot = label_binarize(y, classes=classes_order)
    eps = 1e-15
    ll = -np.log(np.clip((proba * y_onehot).sum(axis=1), eps, 1)).mean()
    brier = ((proba - y_onehot) ** 2).sum(axis=1).mean()
    return ll, brier


validator = WalkForwardValidator(seasons=sorted(all_df["Season"].unique()))

fold_rows = []
all_model_ll, all_market_ll = [], []
all_model_brier, all_market_brier = [], []

for train_idx, val_idx, test_idx, train_seasons, val_season_list, test_season in validator.split(
    all_df, window=WINDOW, val_seasons=VAL_SEASONS
):

    train_df = all_df.loc[train_idx]
    val_df = all_df.loc[val_idx]
    test_df = all_df.loc[test_idx]

    X_train = train_df.drop(columns=drop_cols).fillna(0)
    y_train = train_df["FTR"]
    X_val = val_df.drop(columns=drop_cols).fillna(0)
    y_val = val_df["FTR"]
    X_test = test_df.drop(columns=drop_cols).fillna(0)

    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=4,
        min_samples_leaf=10,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1     
    )
    rf.fit(X_train, y_train)
    classes_order = rf.classes_

    # --- Train / validation log loss & Brier -- the overfitting check ---
    train_ll, train_brier = eval_probs(rf, X_train, y_train, classes_order)
    val_ll, val_brier = eval_probs(rf, X_val, y_val, classes_order)

    # --- Test set: model vs. de-vigged market, same as before ---
    proba = rf.predict_proba(X_test)
    overround = 1 / test_df["AvgHomeOdds"] + 1 / test_df["AvgDrawOdds"] + 1 / test_df["AvgAwayOdds"]
    fair = {
        "H": (1 / test_df["AvgHomeOdds"]) / overround,
        "D": (1 / test_df["AvgDrawOdds"]) / overround,
        "A": (1 / test_df["AvgAwayOdds"]) / overround,
    }
    fair_market_proba = np.column_stack([fair[c].values for c in classes_order])

    y_true = test_df["FTR"].values
    y_onehot = label_binarize(y_true, classes=classes_order)

    eps = 1e-15
    model_ll = -np.log(np.clip((proba * y_onehot).sum(axis=1), eps, 1))
    market_ll = -np.log(np.clip((fair_market_proba * y_onehot).sum(axis=1), eps, 1))
    model_brier = ((proba - y_onehot) ** 2).sum(axis=1)
    market_brier = ((fair_market_proba - y_onehot) ** 2).sum(axis=1)

    all_model_ll.append(model_ll)
    all_market_ll.append(market_ll)
    all_model_brier.append(model_brier)
    all_market_brier.append(market_brier)

    fold_rows.append({
        "test_season": test_season,
        "train_seasons": train_seasons,
        "val_season": val_season_list,
        "n_test": len(test_df),
        "train_log_loss": train_ll,
        "val_log_loss": val_ll,
        "test_log_loss": model_ll.mean(),
        "market_log_loss": market_ll.mean(),
        "train_brier": train_brier,
        "val_brier": val_brier,
        "test_brier": model_brier.mean(),
        "market_brier": market_brier.mean(),
    })

    print(
        f"{test_season} (train {train_seasons[0]}..{train_seasons[-1]}, val {val_season_list[0]}): "
        f"log loss train={train_ll:.4f} val={val_ll:.4f} test={model_ll.mean():.4f} market={market_ll.mean():.4f}"
    )

fold_df = pd.DataFrame(fold_rows)
print("\n=== Per-fold summary (overfitting check: train vs. val vs. test) ===")
print(fold_df[["test_season", "train_log_loss", "val_log_loss", "test_log_loss", "market_log_loss"]].round(4))

# --- Pool every match across every fold's TEST set for the model-vs-market test ---
model_ll_all = np.concatenate(all_model_ll)
market_ll_all = np.concatenate(all_market_ll)
model_brier_all = np.concatenate(all_model_brier)
market_brier_all = np.concatenate(all_market_brier)

print(f"\n=== Pooled across all {len(fold_df)} folds ({len(model_ll_all)} test matches) ===")
print(f"Model log loss:  {model_ll_all.mean():.4f}")
print(f"Market log loss: {market_ll_all.mean():.4f}")
print(f"Model Brier:     {model_brier_all.mean():.4f}")
print(f"Market Brier:    {market_brier_all.mean():.4f}")

print(f"\nMean train log loss:      {fold_df['train_log_loss'].mean():.4f}")
print(f"Mean validation log loss: {fold_df['val_log_loss'].mean():.4f}")
print(
    f"Gap (val - train): {fold_df['val_log_loss'].mean() - fold_df['train_log_loss'].mean():.4f}  "
    "(large gap = overfitting -- the model fits the training seasons far better than unseen ones)"
)

t_stat, p_value = stats.ttest_rel(model_ll_all, market_ll_all)
print(f"\nPaired t-test (log loss, model vs market): t={t_stat:.3f}, p={p_value:.6f}")

diff = model_ll_all - market_ll_all
rng = np.random.default_rng(42)
n = len(diff)
boot_means = np.array([diff[rng.integers(0, n, n)].mean() for _ in range(10000)])
ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
print(f"Bootstrap 95% CI for mean log-loss diff (model - market): [{ci_low:.4f}, {ci_high:.4f}]")

model_wins = (model_ll_all < market_ll_all).sum()
market_wins = (market_ll_all < model_ll_all).sum()
print(f"\nModel better on {model_wins}/{n} matches, market better on {market_wins})/{n} matches")
print(f"Model better on {model_wins/n:.2%} of matches, market better on {market_wins/n:.2%} of matches")

14-15 (train 11-12..12-13, val 13-14): log loss train=0.8774 val=0.9625 test=0.9858 market=0.9725
15-16 (train 12-13..13-14, val 14-15): log loss train=0.8525 val=0.9851 test=1.0345 market=1.0301
16-17 (train 13-14..14-15, val 15-16): log loss train=0.8643 val=1.0379 test=0.9411 market=0.9088
17-18 (train 14-15..15-16, val 16-17): log loss train=0.9024 val=0.9544 test=0.9820 market=0.9398
18-19 (train 15-16..16-17, val 17-18): log loss train=0.8706 val=0.9887 test=0.9372 market=0.8983
19-20 (train 16-17..17-18, val 18-19): log loss train=0.8445 val=0.9228 test=1.0019 market=0.9759
20-21 (train 17-18..18-19, val 19-20): log loss train=0.8414 val=0.9921 test=1.0316 market=1.0096
21-22 (train 18-19..19-20, val 20-21): log loss train=0.8495 val=1.0241 test=0.9899 market=0.9425
22-23 (train 19-20..20-21, val 21-22): log loss train=0.9010 val=0.9957 test=0.9838 market=0.9617
23-24 (train 20-21..21-22, val 22-23): log loss train=0.9026 val=0.9817 test=0.9599 market=0.9168
24-25 (train 21-22..

In [13]:
#SAME MODEL BUT TRAINED ON XG FEATURES AS WELL
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import label_binarize
from scipy import stats

# --- Walk-forward across every available season, with a validation season
#     carved out of each training window so overfitting is visible fold-by-fold
#     (train log loss vs. validation log loss), not just inferred after the fact.
#
#     WINDOW and VAL_SEASONS are the ONLY place to change these -- they're passed
#     through as keyword args below, not re-hardcoded at the call site, to avoid
#     the earlier bug where the default said one thing and the call site silently
#     used another. ---

WINDOW = 3        # total seasons of history pulled into each fold
VAL_SEASONS = 1   # how many of the most-recent seasons in that window are held out for validation

all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons_with_bookies.csv")
all_df = all_df.reset_index(drop=True)

drop_cols = [
    "Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season",
    "AvgHomeOdds", "AvgDrawOdds", "AvgAwayOdds", "NumBookies"
]


class WalkForwardValidator:

    def __init__(self, seasons):
        self.seasons = seasons

    def split(self, meta, window, val_seasons=1):
        unique_seasons = sorted(meta["Season"].unique())
        for i in range(window, len(unique_seasons)):
            window_seasons = unique_seasons[i - window:i]
            train_seasons = window_seasons[:-val_seasons]
            val_season_list = window_seasons[-val_seasons:]
            test_season = unique_seasons[i]

            train_idx = meta[meta["Season"].isin(train_seasons)].index
            val_idx = meta[meta["Season"].isin(val_season_list)].index
            test_idx = meta[meta["Season"] == test_season].index

            yield (train_idx, val_idx, test_idx, train_seasons, val_season_list, test_season)


def eval_probs(rf, X, y, classes_order):
    """Log loss + Brier score for a fitted model against a labeled set."""
    proba = rf.predict_proba(X)
    y_onehot = label_binarize(y, classes=classes_order)
    eps = 1e-15
    ll = -np.log(np.clip((proba * y_onehot).sum(axis=1), eps, 1)).mean()
    brier = ((proba - y_onehot) ** 2).sum(axis=1).mean()
    return ll, brier


validator = WalkForwardValidator(seasons=sorted(all_df["Season"].unique()))

fold_rows = []
all_model_ll, all_market_ll = [], []
all_model_brier, all_market_brier = [], []

for train_idx, val_idx, test_idx, train_seasons, val_season_list, test_season in validator.split(
    all_df, window=WINDOW, val_seasons=VAL_SEASONS
):

    train_df = all_df.loc[train_idx]
    val_df = all_df.loc[val_idx]
    test_df = all_df.loc[test_idx]

    X_train = train_df.drop(columns=drop_cols).fillna(0)
    y_train = train_df["FTR"]
    X_val = val_df.drop(columns=drop_cols).fillna(0)
    y_val = val_df["FTR"]
    X_test = test_df.drop(columns=drop_cols).fillna(0)

    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=4,
        min_samples_leaf=10,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1     
    )
    rf.fit(X_train, y_train)
    classes_order = rf.classes_

    # --- Train / validation log loss & Brier -- the overfitting check ---
    train_ll, train_brier = eval_probs(rf, X_train, y_train, classes_order)
    val_ll, val_brier = eval_probs(rf, X_val, y_val, classes_order)

    # --- Test set: model vs. de-vigged market, same as before ---
    proba = rf.predict_proba(X_test)
    overround = 1 / test_df["AvgHomeOdds"] + 1 / test_df["AvgDrawOdds"] + 1 / test_df["AvgAwayOdds"]
    fair = {
        "H": (1 / test_df["AvgHomeOdds"]) / overround,
        "D": (1 / test_df["AvgDrawOdds"]) / overround,
        "A": (1 / test_df["AvgAwayOdds"]) / overround,
    }
    fair_market_proba = np.column_stack([fair[c].values for c in classes_order])

    y_true = test_df["FTR"].values
    y_onehot = label_binarize(y_true, classes=classes_order)

    eps = 1e-15
    model_ll = -np.log(np.clip((proba * y_onehot).sum(axis=1), eps, 1))
    market_ll = -np.log(np.clip((fair_market_proba * y_onehot).sum(axis=1), eps, 1))
    model_brier = ((proba - y_onehot) ** 2).sum(axis=1)
    market_brier = ((fair_market_proba - y_onehot) ** 2).sum(axis=1)

    all_model_ll.append(model_ll)
    all_market_ll.append(market_ll)
    all_model_brier.append(model_brier)
    all_market_brier.append(market_brier)

    fold_rows.append({
        "test_season": test_season,
        "train_seasons": train_seasons,
        "val_season": val_season_list,
        "n_test": len(test_df),
        "train_log_loss": train_ll,
        "val_log_loss": val_ll,
        "test_log_loss": model_ll.mean(),
        "market_log_loss": market_ll.mean(),
        "train_brier": train_brier,
        "val_brier": val_brier,
        "test_brier": model_brier.mean(),
        "market_brier": market_brier.mean(),
    })

    print(
        f"{test_season} (train {train_seasons[0]}..{train_seasons[-1]}, val {val_season_list[0]}): "
        f"log loss train={train_ll:.4f} val={val_ll:.4f} test={model_ll.mean():.4f} market={market_ll.mean():.4f}"
    )

fold_df = pd.DataFrame(fold_rows)
print("\n=== Per-fold summary (overfitting check: train vs. val vs. test) ===")
print(fold_df[["test_season", "train_log_loss", "val_log_loss", "test_log_loss", "market_log_loss"]].round(4))

# --- Pool every match across every fold's TEST set for the model-vs-market test ---
model_ll_all = np.concatenate(all_model_ll)
market_ll_all = np.concatenate(all_market_ll)
model_brier_all = np.concatenate(all_model_brier)
market_brier_all = np.concatenate(all_market_brier)

print(f"\n=== Pooled across all {len(fold_df)} folds ({len(model_ll_all)} test matches) ===")
print(f"Model log loss:  {model_ll_all.mean():.4f}")
print(f"Market log loss: {market_ll_all.mean():.4f}")
print(f"Model Brier:     {model_brier_all.mean():.4f}")
print(f"Market Brier:    {market_brier_all.mean():.4f}")

print(f"\nMean train log loss:      {fold_df['train_log_loss'].mean():.4f}")
print(f"Mean validation log loss: {fold_df['val_log_loss'].mean():.4f}")
print(
    f"Gap (val - train): {fold_df['val_log_loss'].mean() - fold_df['train_log_loss'].mean():.4f}  "
    "(large gap = overfitting -- the model fits the training seasons far better than unseen ones)"
)

t_stat, p_value = stats.ttest_rel(model_ll_all, market_ll_all)
print(f"\nPaired t-test (log loss, model vs market): t={t_stat:.3f}, p={p_value:.6f}")

diff = model_ll_all - market_ll_all
rng = np.random.default_rng(42)
n = len(diff)
boot_means = np.array([diff[rng.integers(0, n, n)].mean() for _ in range(10000)])
ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
print(f"Bootstrap 95% CI for mean log-loss diff (model - market): [{ci_low:.4f}, {ci_high:.4f}]")

model_wins = (model_ll_all < market_ll_all).sum()
market_wins = (market_ll_all < model_ll_all).sum()
print(f"\nModel better on {model_wins}/{n} matches, market better on {market_wins})/{n} matches")
print(f"Model better on {model_wins/n:.2%} of matches, market better on {market_wins/n:.2%} of matches")

17-18 (train 14-15..15-16, val 16-17): log loss train=0.8945 val=0.9498 test=0.9762 market=0.9398
18-19 (train 15-16..16-17, val 17-18): log loss train=0.8645 val=0.9864 test=0.9397 market=0.8983
19-20 (train 16-17..17-18, val 18-19): log loss train=0.8315 val=0.9197 test=0.9999 market=0.9759
20-21 (train 17-18..18-19, val 19-20): log loss train=0.8313 val=0.9888 test=1.0216 market=1.0096
21-22 (train 18-19..19-20, val 20-21): log loss train=0.8456 val=1.0246 test=0.9837 market=0.9425
22-23 (train 19-20..20-21, val 21-22): log loss train=0.8914 val=0.9865 test=0.9791 market=0.9617
23-24 (train 20-21..21-22, val 22-23): log loss train=0.8878 val=0.9819 test=0.9530 market=0.9168
24-25 (train 21-22..22-23, val 23-24): log loss train=0.8713 val=0.9575 test=0.9967 market=0.9820
25-26 (train 22-23..23-24, val 24-25): log loss train=0.8536 val=1.0020 test=1.0409 market=1.0195

=== Per-fold summary (overfitting check: train vs. val vs. test) ===
  test_season  train_log_loss  val_log_loss  tes

In [6]:
#SAME MODEL BUT TRAINED ON XG FEATURES AS WELL + compared vs BET365
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import label_binarize
from scipy import stats

# --- Walk-forward across every available season, with a validation season
#     carved out of each training window so overfitting is visible fold-by-fold
#     (train log loss vs. validation log loss), not just inferred after the fact.
#
#     WINDOW and VAL_SEASONS are the ONLY place to change these -- they're passed
#     through as keyword args below, not re-hardcoded at the call site, to avoid
#     the earlier bug where the default said one thing and the call site silently
#     used another. ---

WINDOW = 4        # total seasons of history pulled into each fold
VAL_SEASONS = 1   # how many of the most-recent seasons in that window are held out for validation

all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons_with_bookies.csv")
all_df = all_df.reset_index(drop=True)

drop_cols = [
    "Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season",
    "AvgHomeOdds", "AvgDrawOdds", "AvgAwayOdds", "NumBookies", "B365HomeOdds", "B365DrawOdds", "B365AwayOdds"
]


class WalkForwardValidator:

    def __init__(self, seasons):
        self.seasons = seasons

    def split(self, meta, window, val_seasons=1):
        unique_seasons = sorted(meta["Season"].unique())
        for i in range(window, len(unique_seasons)):
            window_seasons = unique_seasons[i - window:i]
            train_seasons = window_seasons[:-val_seasons]
            val_season_list = window_seasons[-val_seasons:]
            test_season = unique_seasons[i]

            train_idx = meta[meta["Season"].isin(train_seasons)].index
            val_idx = meta[meta["Season"].isin(val_season_list)].index
            test_idx = meta[meta["Season"] == test_season].index

            yield (train_idx, val_idx, test_idx, train_seasons, val_season_list, test_season)


def eval_probs(rf, X, y, classes_order):
    """Log loss + Brier score for a fitted model against a labeled set."""
    proba = rf.predict_proba(X)
    y_onehot = label_binarize(y, classes=classes_order)
    eps = 1e-15
    ll = -np.log(np.clip((proba * y_onehot).sum(axis=1), eps, 1)).mean()
    brier = ((proba - y_onehot) ** 2).sum(axis=1).mean()
    return ll, brier


validator = WalkForwardValidator(seasons=sorted(all_df["Season"].unique()))

fold_rows = []
all_model_ll, all_BET365_ll = [], []
all_model_brier, all_BET365_brier = [], []

for train_idx, val_idx, test_idx, train_seasons, val_season_list, test_season in validator.split(
    all_df, window=WINDOW, val_seasons=VAL_SEASONS
):

    train_df = all_df.loc[train_idx]
    val_df = all_df.loc[val_idx]
    test_df = all_df.loc[test_idx]

    X_train = train_df.drop(columns=drop_cols).fillna(0)
    y_train = train_df["FTR"]
    X_val = val_df.drop(columns=drop_cols).fillna(0)
    y_val = val_df["FTR"]
    X_test = test_df.drop(columns=drop_cols).fillna(0)

    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=4,
        min_samples_leaf=10,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1     
    )
    rf.fit(X_train, y_train)
    classes_order = rf.classes_

    # --- Train / validation log loss & Brier -- the overfitting check ---
    train_ll, train_brier = eval_probs(rf, X_train, y_train, classes_order)
    val_ll, val_brier = eval_probs(rf, X_val, y_val, classes_order)

    # --- Test set: model vs. de-vigged market, same as before ---
    proba = rf.predict_proba(X_test)
    overround = 1 / test_df["B365HomeOdds"] + 1 / test_df["B365DrawOdds"] + 1 / test_df["B365AwayOdds"]
    fair = {
        "H": (1 / test_df["B365HomeOdds"]) / overround,
        "D": (1 / test_df["B365DrawOdds"]) / overround,
        "A": (1 / test_df["B365AwayOdds"]) / overround,
    }
    fair_market_proba = np.column_stack([fair[c].values for c in classes_order])

    y_true = test_df["FTR"].values
    y_onehot = label_binarize(y_true, classes=classes_order)

    eps = 1e-15
    model_ll = -np.log(np.clip((proba * y_onehot).sum(axis=1), eps, 1))
    BET365_ll = -np.log(np.clip((fair_market_proba * y_onehot).sum(axis=1), eps, 1))
    model_brier = ((proba - y_onehot) ** 2).sum(axis=1)
    BET365_brier = ((fair_market_proba - y_onehot) ** 2).sum(axis=1)

    all_model_ll.append(model_ll)
    all_BET365_ll.append(BET365_ll)
    all_model_brier.append(model_brier)
    all_BET365_brier.append(BET365_brier)

    fold_rows.append({
        "test_season": test_season,
        "train_seasons": train_seasons,
        "val_season": val_season_list,
        "n_test": len(test_df),
        "train_log_loss": train_ll,
        "val_log_loss": val_ll,
        "test_log_loss": model_ll.mean(),
        "BET365_log_loss": BET365_ll.mean(),
        "train_brier": train_brier,
        "val_brier": val_brier,
        "test_brier": model_brier.mean(),
        "BET365_brier": BET365_brier.mean(),
    })

    print(
        f"{test_season} (train {train_seasons[0]}..{train_seasons[-1]}, val {val_season_list[0]}): "
        f"log loss train={train_ll:.4f} val={val_ll:.4f} test={model_ll.mean():.4f} BET365={BET365_ll.mean():.4f}"
    )

fold_df = pd.DataFrame(fold_rows)
print("\n=== Per-fold summary (overfitting check: train vs. val vs. test) ===")
print(fold_df[["test_season", "train_log_loss", "val_log_loss", "test_log_loss", "BET365_log_loss"]].round(4))

# --- Pool every match across every fold's TEST set for the model-vs-market test ---
model_ll_all = np.concatenate(all_model_ll)
BET365_ll_all = np.concatenate(all_BET365_ll)
model_brier_all = np.concatenate(all_model_brier)
BET365_brier_all = np.concatenate(all_BET365_brier)

print(f"\n=== Pooled across all {len(fold_df)} folds ({len(model_ll_all)} test matches) ===")
print(f"Model log loss:  {model_ll_all.mean():.4f}")
print(f"BET365 log loss: {BET365_ll_all.mean():.4f}")
print(f"Model Brier:     {model_brier_all.mean():.4f}")
print(f"BET365 Brier:    {BET365_brier_all.mean():.4f}")

print(f"\nMean train log loss:      {fold_df['train_log_loss'].mean():.4f}")
print(f"Mean validation log loss: {fold_df['val_log_loss'].mean():.4f}")
print(
    f"Gap (val - train): {fold_df['val_log_loss'].mean() - fold_df['train_log_loss'].mean():.4f}  "
    "(large gap = overfitting -- the model fits the training seasons far better than unseen ones)"
)

t_stat, p_value = stats.ttest_rel(model_ll_all, BET365_ll_all)
print(f"\nPaired t-test (log loss, model vs BET365): t={t_stat:.3f}, p={p_value:.6f}")

diff = model_ll_all - BET365_ll_all
rng = np.random.default_rng(42)
n = len(diff)
boot_means = np.array([diff[rng.integers(0, n, n)].mean() for _ in range(10000)])
ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
print(f"Bootstrap 95% CI for mean log-loss diff (model - BET365): [{ci_low:.4f}, {ci_high:.4f}]")

model_wins = (model_ll_all < BET365_ll_all).sum()
BET365_wins = (BET365_ll_all < model_ll_all).sum()
print(f"\nModel better on {model_wins}/{n} matches, BET365 better on {BET365_wins}/{n} matches")
print(f"Model better on {model_wins/n:.2%} of matches, BET365 better on {BET365_wins/n:.2%} of matches")

18-19 (train 14-15..16-17, val 17-18): log loss train=0.8837 val=0.9673 test=0.9361 BET365=0.8977
19-20 (train 15-16..17-18, val 18-19): log loss train=0.8703 val=0.9315 test=0.9980 BET365=0.9763
20-21 (train 16-17..18-19, val 19-20): log loss train=0.8345 val=1.0000 test=1.0322 BET365=1.0121
21-22 (train 17-18..19-20, val 20-21): log loss train=0.8619 val=1.0305 test=0.9763 BET365=0.9435
22-23 (train 18-19..20-21, val 21-22): log loss train=0.8852 val=0.9799 test=0.9721 BET365=0.9623
23-24 (train 19-20..21-22, val 22-23): log loss train=0.9007 val=0.9807 test=0.9476 BET365=0.9164
24-25 (train 20-21..22-23, val 23-24): log loss train=0.8937 val=0.9494 test=1.0035 BET365=0.9822
25-26 (train 21-22..23-24, val 24-25): log loss train=0.8726 val=1.0089 test=1.0405 BET365=1.0224

=== Per-fold summary (overfitting check: train vs. val vs. test) ===
  test_season  train_log_loss  val_log_loss  test_log_loss  BET365_log_loss
0       18-19          0.8837        0.9673         0.9361           0